# 20 — Paper Figures: Pipeline Diagram

A schematic of this project's end-to-end preprocessing-and-classification
pipeline, in the box-and-arrow grammar of
`../SEP_DataAugmentation 2/paper/figures/pipeline.png` (that project's
Figure 2), adapted to this project's own five stages: Hybrid Normalization,
Borderline Cleaning (Tomek Links), Random Under-Sampling, Over-Sampling
(SMOTE / ADASYN / TimeGAN / Diffusion), and the four classifiers.

Every number drawn in this figure is quoted from the project's own
notebooks and `results/` files rather than recomputed here.

**Writes:** `./paper/figures/pipeline.pdf` and `.png`.
**Does not touch:** notebooks 11-19 or their output.


## 1. Build the Diagram

In [1]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle, FancyArrowPatch

FIG_DIR = "./paper/Figures"
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
})

# Okabe-Ito-derived, colorblind-safe accents. Body content is mostly BLACK
# for legibility; the panel's own accent colors the structural labels
# ('why', 'compared with') and the one or two things worth highlighting
# per panel (the method actually used, or a meaningful family split).
BLUE, VERM, TEAL, GOLD = "#0072B2", "#D55E00", "#009E73", "#CC9A00"
# Two more Okabe-Ito accents, reserved for the classical/generative split
# inside panel IV specifically -- distinct from every panel's own accent
# (including that panel's own TEAL) so the family split doesn't blend
# into, or get mistaken for, another panel's identity color.
SKY, PLUM = "#56B4E9", "#CC79A7"
# Training's own accent -- distinct from the green already used for the
# Over-Sampling half of the merged step III box, so Training's "why" /
# "models used" labels and its GRU highlight don't read as related to it.
ORANGE = "#E69F00"
BLACK = "#1A1A1A"
EDGE = "0.62"
FS = 1.16  # global font-size multiplier applied on top of every literal size below

W, H = 7.16, 2.55
fig = plt.figure(figsize=(W, H))
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, W)
ax.set_ylim(0, H)
ax.set_aspect("equal")
ax.axis("off")

BY0, BY1, TBAR = 0.20, H - 0.10, 0.34
gap = 0.16
n_panels = 5
avail = W - gap * (n_panels + 1)
# Layout keeps the original 5-slot grid so every piece of text below still
# sits at its already-verified, non-overrunning position; "rus" and "os"
# just share one outer box now instead of each getting their own.
KEYS = ["norm", "clean", "rus", "os", "clf"]
pw = avail / n_panels
P = {k: (gap + i * (pw + gap), gap + i * (pw + gap) + pw) for i, k in enumerate(KEYS)}
checks = []


def panel(key, title, accent):
    x0, x1 = P[key]
    ax.add_patch(FancyBboxPatch((x0, BY0), x1 - x0, BY1 - BY0,
                                 boxstyle="round,pad=0,rounding_size=0.05",
                                 linewidth=0.7, edgecolor=EDGE,
                                 facecolor="white", zorder=1))
    ax.add_patch(Rectangle((x0, BY1 - TBAR), x1 - x0, TBAR,
                            facecolor=accent, alpha=0.13, lw=0, zorder=1.5))
    t = ax.text((x0 + x1) / 2, BY1 - TBAR / 2, title, ha="center",
                va="center", fontsize=6.3 * FS, color=accent, zorder=3,
                linespacing=1.3)
    checks.append((t, key))
    return x0, x1


def put(key, y, txt, size=5.7, color=BLACK, weight="normal"):
    x0, x1 = P[key]
    t = ax.text((x0 + x1) / 2, y, txt, ha="center", va="center",
                fontsize=size * FS, color=color, zorder=3, linespacing=1.25,
                fontweight=weight)
    checks.append((t, key))


def label(key, y, txt, color):
    """Small italic structural tag ('why', 'compared with'), colored
    with the panel's own accent -- distinct from the black body text
    it introduces, so the eye can find each panel's rhythm at a glance."""
    put(key, y, txt, 4.3, color)
    ax.texts[-1].set_style("italic")


# ------------------------------------------------- I. Hybrid Normalization
panel("norm", "I. Hybrid\nNormalization", BLUE)
put("norm", 1.90, "rescales each channel so", 4.9)
put("norm", 1.74, "no instrument dominates", 4.9)
label("norm", 1.50, "why", BLUE)
put("norm", 1.34, "sources differ by orders", 4.6)
put("norm", 1.20, "of magnitude in scale", 4.6)
label("norm", 0.94, "compared with", BLUE)
put("norm", 0.76, "Z-score, Min-Max, Log", 4.5)
put("norm", 0.50, "used:  Hybrid", 5.4, BLUE, "bold")

# -------------------------------------------------- II. Borderline Cleaning
panel("clean", "II. Borderline\nCleaning", GOLD)
put("clean", 1.90, "removes NSEP samples that", 4.4)
put("clean", 1.74, "sit on the SEP boundary", 4.9)
label("clean", 1.50, "why", GOLD)
put("clean", 1.34, "ambiguous NSEP samples", 4.6)
put("clean", 1.20, "mislead the classifier", 4.6)
label("clean", 0.94, "compared with", GOLD)
put("clean", 0.76, "ENN, NearMiss-3", 4.5)
put("clean", 0.50, "used:  Tomek Links", 5.4, GOLD, "bold")

# ---------------------------- III. Random Under-Sampling & Over-Sampling
# One shared box around the old "rus" and "os" slots -- the under- and
# over-sampling steps are two halves of a single resampling stage, so
# they're presented as one numbered step with a divider instead of two.
us_x0, us_x1 = P["rus"][0], P["os"][1]
us_mid = (us_x0 + us_x1) / 2
P["us"] = (us_x0, us_x1)
ax.add_patch(FancyBboxPatch((us_x0, BY0), us_x1 - us_x0, BY1 - BY0,
                             boxstyle="round,pad=0,rounding_size=0.05",
                             linewidth=0.7, edgecolor=EDGE,
                             facecolor="white", zorder=1))
ax.add_patch(Rectangle((us_x0, BY1 - TBAR), us_mid - us_x0, TBAR,
                        facecolor=VERM, alpha=0.13, lw=0, zorder=1.5))
ax.add_patch(Rectangle((us_mid, BY1 - TBAR), us_x1 - us_mid, TBAR,
                        facecolor=TEAL, alpha=0.13, lw=0, zorder=1.5))
ax.plot([us_mid, us_mid], [BY0 + 0.06, BY1 - TBAR - 0.05], color=EDGE,
        lw=0.7, zorder=2)
us_title = ax.text(us_mid, BY1 - TBAR / 2, "III. Random Under-Sampling & Over-Sampling",
                    ha="center", va="center", fontsize=6.0 * FS, color=BLACK,
                    zorder=3, linespacing=1.3)
checks.append((us_title, "us"))

put("rus", 1.94, "randomly drops NSEP", 4.9)
put("rus", 1.80, "samples; all SEP kept", 4.9)
label("rus", 1.60, "levels tried", VERM)
bx0, bx1 = P["rus"]
bcx = (bx0 + bx1) / 2
bar_y = 0.96
heights = [0.34, 0.16, 0.07]
bar_labels = ["8000", "2000", "500"]
bw = 0.12
start = bcx - 1.5 * bw - 0.06
for i, (hgt, lab) in enumerate(zip(heights, bar_labels)):
    bxp = start + i * (bw + 0.06)
    ax.add_patch(Rectangle((bxp, bar_y), bw, hgt, facecolor=VERM, alpha=0.55
                            - 0.12 * i, lw=0, zorder=3))
    ax.text(bxp + bw / 2, bar_y + hgt + 0.06, lab, fontsize=4.8 * FS, ha="center",
            color=BLACK)
    checks.append((ax.texts[-1], "rus"))
label("rus", 0.60, "why", VERM)
put("rus", 0.44, "prepares matched majority", 4.6)
put("rus", 0.32, "levels for over-sampling", 4.6)

put("os", 1.94, "generates synthetic SEP", 4.9)
put("os", 1.80, "sequences from real ones", 4.9)
label("os", 1.60, "why", TEAL)
put("os", 1.44, "only 118 real SEP", 4.6)
put("os", 1.32, "sequences to train on", 4.6)
label("os", 1.08, "families tried", TEAL)
put("os", 0.92, "SMOTE, ADASYN", 4.8, SKY, "bold")
put("os", 0.76, "TimeGAN, Diffusion", 4.8, PLUM, "bold")
label("os", 0.52, "applied at", TEAL)
put("os", 0.34, "each NSEP level from above", 4.1)

# ------------------------------------------------------------- IV. Training
panel("clf", "IV. Training", BLACK)
put("clf", 1.94, "trains four architectures", 4.9)
put("clf", 1.80, "on every preprocessed set", 4.9)
label("clf", 1.60, "why", ORANGE)
put("clf", 1.44, "checks gains hold across", 4.6)
put("clf", 1.32, "model families, not just one", 4.6)
label("clf", 1.06, "models used", ORANGE)
put("clf", 0.90, "SVM (catch22), PatchTST,", 4.7)
put("clf", 0.76, "InceptionTime", 4.7)
put("clf", 0.58, "GRU", 5.6, ORANGE, "bold")

# -------------------------------------------------------------- arrows
mid_y = (BY0 + BY1 - TBAR) / 2 + 0.02
ARROW_PAIRS = [("norm", "clean"), ("clean", "rus"), ("os", "clf")]
for a, b in ARROW_PAIRS:
    xa = P[a][1]
    xb = P[b][0]
    ax.add_patch(FancyArrowPatch((xa + 0.02, mid_y), (xb - 0.02, mid_y),
                                  arrowstyle="-|>", mutation_scale=7, lw=0.9,
                                  color="0.35", zorder=4))

fig.canvas.draw()
overrun = []
for t, key in checks:
    x0, x1 = P[key]
    bb = t.get_window_extent(renderer=fig.canvas.get_renderer())
    bb_data = bb.transformed(ax.transData.inverted())
    if bb_data.x0 < x0 - 0.03 or bb_data.x1 > x1 + 0.03:
        overrun.append((key, t.get_text()))
if overrun:
    print("WARNING - text may overrun its panel:", overrun)
else:
    print("No text overruns detected.")

# Manual vertical-collision spot check for panel "rus", where a bar
# chart and text share the same column (the automatic check above only
# catches text running past its panel's left/right edges, not text
# overlapping other text or graphics within the panel).
rus_top_of_bars = bar_y + max(heights) + 0.06 + 0.10  # + label height
print(f"rus: bottom of intro text = 1.80-0.10=1.70, top of bars+label "
      f"= {rus_top_of_bars:.2f} -> {'OK, clear' if rus_top_of_bars < 1.70 else 'WARNING: may overlap'}")

fig.savefig(f"{FIG_DIR}/pipeline.pdf")
fig.savefig(f"{FIG_DIR}/pipeline.png", dpi=400)
plt.show()
print("Saved pipeline.pdf/png")

No text overruns detected.
rus: bottom of intro text = 1.80-0.10=1.70, top of bars+label = 1.46 -> OK, clear


Saved pipeline.pdf/png


/var/folders/fx/gjhbmrbj5jn295_9wrqpbsv80000gn/T/ipykernel_64633/2286670698.py:207: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
